# Step 3: Optimization & Performance Improvement

**Goal:** address class imbalance with SMOTE and aim for the high
performance benchmark of `ishaq2021improving` — Ishaq et al. (2021), *IEEE
Access* — Extra Trees + SMOTE, Accuracy 92.6%.

::: danger Red Flag — exactly as stated in the original assignment
SMOTE must **only ever be applied to the Train set**. If Test is also
SMOTE'd, the model is evaluated on synthetic (fake) data instead of real
patients — completely invalidating the clinical meaning of the result.

In [1]:
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

try:
    import imblearn  # noqa: F401
except ImportError:
    import subprocess, sys as _sys
    print("[i] imbalanced-learn not found in this environment (Colab/Kaggle) — installing...")
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn'], check=True)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score, classification_report
from imblearn.over_sampling import SMOTE

def _load_heart_failure_data():
    """Load the local file (../data/...) if present (running inside a
    cloned HMYT repo); otherwise (opened standalone via Colab/Kaggle, no
    accompanying data/ folder) automatically download it from the public
    mirror on hmyt-book (Public repo, verified 2026-09-23)."""
    import os
    local_path = "../data/heart_failure_clinical_records_dataset.csv"
    remote_url = ("https://raw.githubusercontent.com/fossbk-spec/hmyt-book/gh-pages/"
                  "labs_chuyen_de/ch02_suy_tim_risk_dxai/data/"
                  "heart_failure_clinical_records_dataset.csv")
    path = local_path if os.path.exists(local_path) else remote_url
    if path == remote_url:
        print(f"[i] Local data not found — downloading from the public mirror:\n    {remote_url}")
    return pd.read_csv(path).rename(columns={'death_event': 'DEATH_EVENT'})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = _load_heart_failure_data()
FEATURE_COLS = [c for c in df.columns if c != 'DEATH_EVENT']
X, y = df[FEATURE_COLS], df['DEATH_EVENT'].values

# Keep the SAME Train/Test split as Steps 1-2 for a fair comparison
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)   # Test stays UNCHANGED, NOT run through SMOTE

print(f"Before SMOTE — Train: {len(y_train)} samples, death rate {y_train.mean()*100:.1f}%")


Before SMOTE — Train: 239 samples, death rate 32.2%


## 1. Applying SMOTE — Train Set ONLY

In [2]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_s, y_train)

print(f"After SMOTE — Train: {len(y_train_smote)} samples, death rate {y_train_smote.mean()*100:.1f}%")
print(f"(Classes balanced: {np.sum(y_train_smote==0)} survived / {np.sum(y_train_smote==1)} died)")
print(f"\nTest set UNCHANGED: {len(y_test)} samples, death rate {y_test.mean()*100:.1f}% (real distribution preserved)")

After SMOTE — Train: 324 samples, death rate 50.0%
(Classes balanced: 162 survived / 162 died)

Test set UNCHANGED: 60 samples, death rate 31.7% (real distribution preserved)


## 2. Training Extra Trees on the SMOTE'd Data

In [3]:
etc = ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE)
etc.fit(X_train_smote, y_train_smote)
y_pred_etc = etc.predict(X_test_s)   # Evaluated on the REAL Test set, not run through SMOTE

acc_etc = accuracy_score(y_test, y_pred_etc)
mcc_etc = matthews_corrcoef(y_test, y_pred_etc)
f1_etc = f1_score(y_test, y_pred_etc)
print(f"Extra Trees + SMOTE -> Accuracy: {acc_etc:.4f} | MCC: {mcc_etc:.4f} | F1: {f1_etc:.4f}")
print()
print(classification_report(y_test, y_pred_etc, target_names=['Survived', 'Death']))

Extra Trees + SMOTE -> Accuracy: 0.7667 | MCC: 0.4182 | F1: 0.5333

              precision    recall  f1-score   support

    Survived       0.78      0.93      0.84        41
       Death       0.73      0.42      0.53        19

    accuracy                           0.77        60
   macro avg       0.75      0.67      0.69        60
weighted avg       0.76      0.77      0.75        60



## 3. With SMOTE vs. Without SMOTE (same Extra Trees algorithm)

In [4]:
etc_no_smote = ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE)
etc_no_smote.fit(X_train_s, y_train)   # No SMOTE, trained directly on the imbalanced Train set
y_pred_no_smote = etc_no_smote.predict(X_test_s)

acc_no = accuracy_score(y_test, y_pred_no_smote)
mcc_no = matthews_corrcoef(y_test, y_pred_no_smote)
f1_no = f1_score(y_test, y_pred_no_smote)

print("=== WITH SMOTE vs. WITHOUT SMOTE (same Extra Trees, same Test set) ===")
print(f"{'Configuration':<28}{'Accuracy':>12}{'MCC':>10}{'F1':>10}")
print(f"{'Extra Trees (no SMOTE)':<28}{acc_no:>12.4f}{mcc_no:>10.4f}{f1_no:>10.4f}")
print(f"{'Extra Trees + SMOTE':<28}{acc_etc:>12.4f}{mcc_etc:>10.4f}{f1_etc:>10.4f}")
print(f"{'ishaq2021improving (paper)':<28}{'0.9260':>12}{'-':>10}{'-':>10}")
print(f"\nAccuracy delta vs. the literature: {acc_etc - 0.926:+.4f}")

=== WITH SMOTE vs. WITHOUT SMOTE (same Extra Trees, same Test set) ===
Configuration                   Accuracy       MCC        F1
Extra Trees (no SMOTE)            0.7500    0.3685    0.4828
Extra Trees + SMOTE               0.7667    0.4182    0.5333
ishaq2021improving (paper)        0.9260         -         -

Accuracy delta vs. the literature: -0.1593


## 4. Step 3 Summary

The numbers in the two cells above are **actual run results**, not
hard-coded to match the literature. If Accuracy has not approached
92.6%, valid directions for improvement (without changing the Test scope,
without SMOTE-ing Test) include: increasing `n_estimators`, trying
`RandomForestClassifier`/`GradientBoostingClassifier` instead of Extra
Trees, or combining `SMOTETomek`/`SMOTEENN` (borderline-noise filtering
after sample generation) — in the spirit of the "reduced ablation study"
called for in the original assignment.

**Next:** [`4_xai.ipynb`](./4_xai.ipynb) — Step 4, explaining the model
with SHAP and comparing Serum Creatinine / Ejection Fraction.